In [ ]:
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate,SystemMessagePromptTemplate,HumanMessagePromptTemplate
from langchain_core.documents import Document
from typing import List,TypedDict
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

In [ ]:
from langgraph.graph import StateGraph,START,END

In [ ]:
model= ChatOllama(model="qwen2.5:0.5b")

In [ ]:
docs = (
    PyPDFLoader("docs/dl_fundamentals.pdf").load() + 
    PyPDFLoader("docs/Intro_Machine_Learning.pdf").load()
    )
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(docs  )

for d in chunks:
    d.page_content = d.page_content.encode("utf-8","ignore").decode("utf-8","ignore")


embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L12-v2")
vector_store = FAISS.from_documents(chunks,embeddings)
retriever = vector_store.as_retriever(search_type="similarity",search_kwargs={'k':4})

In [ ]:
UPPER_THRESHOLD = 0.8
LOWER_THRESHOLD = 0.3

In [ ]:
class State(TypedDict):
    question : str 
    docs : List[Document]

    best_docs : List[Document]
    decide : str
    reason : str 

    strips : List[str]
    imp_stips : List[str]
    refined_context: str

    web_search_docs : List[Document]
    web_query: str
    answer: str 


In [ ]:
def retriever_docs(state:State)->State:
    docs = retriever.invoke(state["question"])
    return {"docs":docs}

In [ ]:
from pydantic import BaseModel

In [ ]:
class Evaluate_docs(BaseModel):
    score:float
    reason:str

doc_eval_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("""
        You are a document evaluator. You will receive **one retrieved chunk and a question**.
        For a chunk:
        1. Assess how relevant and helpful this chunk is for answering the given question.
        2. Assign a **score** between 0.0 (irrelevant) and 1.0 (highly relevant/complete).
        3. Provide a **concise reason** explaining why the score was given.
        4. Output **ONLY valid JSON** in this format:
        {{
            "score": <float>,
            "reason": "<string>"
        }}
        5. Be precise, strict, and do not include any text outside the JSON.
        Example output:
        {{
            "score": 0.8,
            "reason": "This chunk contains key information about attention mechanisms relevant to the question."
        }}
    """),
    HumanMessagePromptTemplate.from_template("Question: {question}\n\nContext: {context}\n\nAnswer:")
])
    
docs_eval_chain = doc_eval_prompt | model.with_structured_output(Evaluate_docs)


def eval_each_doc(state:State)->State:
    
    q = state["question"]
    scores = []
    reasons : List[str] =[]
    good_docs : List[Document] = []


    for d in state["docs"]:
        r = docs_eval_chain.invoke({"question": q, "context": d.page_content})
        scores.append(r.score)
        reasons.append(r.reason)


        if r.score > LOWER_THRESHOLD:
            good_docs.append(d)

    #correct if any one doc is above threshold
    if any(s>UPPER_THRESHOLD for s in scores):
        return {
            "best_docs":good_docs,
            "decide":"CORRECT",
            "reason":f"at least one retrieved chunk > {reasons[scores.index(max(scores))]}"
        }
    
    if len(scores)>0 and all(s<LOWER_THRESHOLD for s in scores):
        why ="any of the retrieved chunks is irrelevant"
        return {
            "best_docs":[],
            "decide":"INCORRECT",
            "reason":f"all retrieved chunks < {reasons[scores.index(min(scores))]} & {why}"
        }
    why = "mixed releavant chunks"
    
    return {
            "best_docs":good_docs,
            "decide":"AMBIGUOUS",
            "reason":f"no chunks score>{UPPER_THRESHOLD} and no chunks score<{LOWER_THRESHOLD} & {why}"
        }
    

In [ ]:
#sentece level strips
from typing import List
import re
def sentence_strips(text:str)->List[str]:
    text = re.sub(r'\s+', ' ', text).strip()
    sentence = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', text)

    return [s.strip() for s in sentence if len(s.strip())>30]


class imp_yes_no(BaseModel):
    keep : bool 

filter_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "You are a strict relevance filter. "
        "Decide if a document is relevant to the question. "
        "Return keep=true if relevant, else keep=false. "
        "Output JSON only."
    ),
    HumanMessagePromptTemplate.from_template(
        "Question: {question}\n\nContext: {context}\n\nAnswer:"
    )
])

chain_seq = filter_prompt | model.with_structured_output(imp_yes_no)

def refine(state:State)->State :

    q = state["question"]

    if state.get("decide")=="CORRECT" :
        context = "\n\n".join(d.page_content for d in state["best_docs"]).strip()

    elif state.get("decide")=="INCORRECT":
        context = "\n\n".join(d.page_content for d in state["web_search_docs"]).strip()

    elif state.get("decide")=="AMBIGUOUS":
        docs_to_use = state["good_docs"] + state["web_search_docs"]
        context = "\n\n".join(d.page_content for d in docs_to_use).strip()

    
    strips = sentence_strips(context)

    imp_strips: List[str] = []
    for s in strips:
        res = chain_seq.invoke({"question":q,"context":s})
        if res.keep:
            imp_strips.append(s)

    return {
        "strips":strips,
        "imp_stips":imp_strips,
        "refined_context": "\n\n".join(imp_strips).strip()
    }


In [ ]:
#web search tavily

class WebQueryWrite(BaseModel):
    query : str

query_rewrite_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        """
    You are a query rewriting assistant for web search.

    Your task:
    Convert the user's question into a concise, high-quality search engine query.

    STRICT RULES:
    1. Output ONLY the search query (no explanations, no sentences).
    2. Use short keyword phrases (like Google search).
    3. Remove filler words (e.g., "what is", "tell me about").
    4. Keep important entities (names, companies, topics).
    5. If the question implies recency (e.g., "latest", "recent", "today", "last week"):
    → Add a time constraint like: "last 7 days" or "last 30 days".
    6. Do NOT answer the question.
    7. Do NOT include punctuation except necessary words.
    8. Keep it under 10 words whenever possible.
    9. return JSON with a single key: query

    Examples:
    Question: What is a neural network?
    Output: neural network explanation basics

    Question: latest AI news
    Output: AI news last 7 days

    Question: what is the stock price of ONGC?
    Output: ONGC stock price today

    Question: recent breakthroughs in LLMs
    Output: LLM breakthroughs last 30 days
"""
    ),
    HumanMessagePromptTemplate.from_template(
        "Question: {question}"
    )
])

rewrite_query_chain = query_rewrite_prompt | model.with_structured_output(WebQueryWrite)


def rewrite_query_node(state:State):

    res = rewrite_query_chain.invoke({"question":state["question"]})
    return {"web_query":res}


from langchain_community.tools.tavily_search import TavilySearchResults

tool = TavilySearchResults(max_results=5)

def web_search_node(state:State)->State:

    q = state.get("web_query") or state.get("question") 

    res= tool.invoke({"query":q})

    web_docs = []

    for r in res or [] :
        title = r.get("title","")
        url = r.get("url","")
        content = r.get("content","") or r.get("snippet","")


        text = f"Title: {title}\nURL: {url}\nContent: {content}"
        web_docs.append(Document(page_content=text,metadata={"url":url,"title":title}))


    return {
        "web_search_docs":web_docs
    }


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
prompt = ChatPromptTemplate.from_messages([
    {"role": "system", "content": """You are a helpful assistant. Answer only from the context.
                                    If you don't know the answer or context is empty,
                                    just say that you don't know, don't try to make up an answer."""},
    {"role": "human", "content": "Question: {question}\n\nContext: {context}\n\nAnswer:"}
])
def generate(state:State)->State:
    context = state["refined_context"]
    return {"answer": model.invoke(prompt.format(question=state["question"], context=context))}

In [ ]:

def route_after_eval(state:State)->State:

    if(state["decide"]=="CORRECT"):
        return "refine"
    else:
        return "rewrite_query"

In [ ]:
graph = StateGraph(
    State
)

graph.add_node("retrieve_docs",retriever_docs)
graph.add_node("eval_each_doc",eval_each_doc)
graph.add_node("refine",refine)
graph.add_node("rewrite_query",rewrite_query_node)
graph.add_node("generate",generate)
graph.add_node("web_search",web_search_node)



graph.add_edge(START,"retrieve_docs")
graph.add_edge("retrieve_docs","eval_each_doc")

graph.add_conditional_edges("eval_each_doc",route_after_eval,{
    "refine":"refine",  
    "rewrite_query":"rewrite_query" ,  
    }
    )
graph.add_edge("rewrite_query","web_search")
graph.add_edge("web_search","refine")
graph.add_edge("refine","generate")
graph.add_edge("generate",END)



workflow= graph.compile()
workflow

In [ ]:
#example

question2 = "what's the latest AI news from past week?"
res2 = workflow.invoke({"question":question2})

print(res2["decide"])
print("*"*20)
print(res2["reason"])
print("*"*20)
print(res2["answer"])

In [ ]:
#example


question4 = "what's the  stock price of ONGC?"
res4 = workflow.invoke({"question":question4})


print(res4["decide"])
print("*"*20)
print(res4["reason"])
print("*"*20)
print(res4["answer"])